# Étape 1-2 : Inspection et Nettoyage des Données

Chargement, exploration et nettoyage du dataset immobilier de Nouakchott.

In [10]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('whitegrid')

from src.data_cleaning import load_data, clean_dataframe, get_missing_summary, get_basic_stats

TRAIN_PATH = '../data/raw/kaggle_train.csv'
TEST_PATH  = '../data/raw/kaggle_test.csv'

## Étape 1 : Chargement et inspection initiale

In [11]:
train_raw, test_raw = load_data(TRAIN_PATH, TEST_PATH)
print(f"Train : {train_raw.shape}")
print(f"Test  : {test_raw.shape}")
print(f"\n'prix' dans train : {'prix' in train_raw.columns}")
print(f"'prix' dans test  : {'prix' in test_raw.columns}")

Train : (1153, 12)
Test  : (289, 11)

'prix' dans train : True
'prix' dans test  : False


In [12]:
print("=== Types et valeurs manquantes (TRAIN) ===")
print(train_raw.dtypes)
print("\n=== Describe (TRAIN) ===")
train_raw.describe()

=== Types et valeurs manquantes (TRAIN) ===
id                    int64
titre                   str
prix                float64
surface_m2          float64
nb_chambres         float64
nb_salons           float64
nb_sdb              float64
quartier                str
description             str
caracteristiques        str
source                  str
date_publication        str
dtype: object

=== Describe (TRAIN) ===


,id,prix,surface_m2,nb_chambres,nb_salons,nb_sdb
count,1153.000000,1.153000e+03,1153.000000,1139.000000,1153.000000,320.000000
mean,723.044232,4.339931e+06,254.520382,4.577700,2.259324,2.390625
std,414.245127,5.451494e+06,178.915730,2.325807,1.766161,1.502529
min,0.000000,2.000000e+05,20.000000,0.000000,0.000000,0.000000
25%,368.000000,1.300000e+06,150.000000,3.000000,1.000000,2.000000
50%,729.000000,2.600000e+06,200.000000,4.000000,2.000000,2.000000
75%,1072.000000,5.800000e+06,300.000000,6.000000,3.000000,3.000000
max,1438.000000,5.400000e+07,2000.000000,15.000000,32.000000,12.000000


In [13]:
train_raw.head()

,id,titre,prix,surface_m2,nb_chambres,nb_salons,nb_sdb,quartier,description,caracteristiques,source,date_publication
0,1076,منزل احذ اللنكات حمام الياسمين,1800000.0,150.0,3.0,2.0,NaN,Arafat,دار للبيع اعل شارع اكبير احذ حمام الياسمين الل...,Titre foncier | 1 balcon(s) | Taille rue: 15.0...,voursa.com,2025-09-13
1,875,فرصة دار مكونه من طابقين ارضي و واحد فوقوني كا...,1800000.0,300.0,6.0,3.0,NaN,Tevragh Zeina,فرصة دار ، الطابق الأرضي يحتاج ترميم بسبب المل...,1 balcon(s) | Taille rue: N/A | Proche de: كرف...,voursa.com,2025-07-06
2,453,دار فتيارت فاتح فبرك,900000.0,216.0,1.0,1.0,NaN,Teyarett,سعر 9ملايين مدخله 50الف حد شاري ول عندو طلب تل...,Titre foncier | type de propriété: Autre | 1 b...,voursa.com,2026-01-20
3,987,دار للبيـــــــــــــــــــــع أفي عين الطلح,1600000.0,150.0,3.0,1.0,2.0,Teyarett,السلام عليكم ذاك دار للبيـــــــــــــــــــــ...,NaN,voursa.com,2025-12-09
4,252,ملح سكتير 2,800000.0,180.0,3.0,2.0,NaN,Toujounine,السلام عليكم \nفرصة للبيـــــــــــــــــــــع...,Titre foncier | 1 balcon(s) | Taille rue: 25.0...,voursa.com,2025-04-18


In [14]:
print("=== Quartiers (TRAIN) ===")
print(train_raw['quartier'].value_counts())
print("\n=== Sources ===")
print(train_raw['source'].value_counts())
print("\n=== Plage de dates ===")
print(train_raw['date_publication'].describe())

=== Quartiers (TRAIN) ===
quartier
Tevragh Zeina    373
Teyarett         270
Arafat           244
Toujounine       115
Dar Naim          82
Ksar              46
Riyadh            13
Sebkha            10
Name: count, dtype: int64

=== Sources ===
source
voursa.com    1153
Name: count, dtype: int64

=== Plage de dates ===
count           1153
unique           329
top       2026-02-26
freq              13
Name: date_publication, dtype: object


In [15]:
# Colonnes numériques vs catégorielles
num_cols = train_raw.select_dtypes(include='number').columns.tolist()
cat_cols = train_raw.select_dtypes(include='object').columns.tolist()
print(f"Colonnes numériques : {num_cols}")
print(f"Colonnes catégorielles : {cat_cols}")

Colonnes numériques : ['id', 'prix', 'surface_m2', 'nb_chambres', 'nb_salons', 'nb_sdb']
Colonnes catégorielles : ['titre', 'quartier', 'description', 'caracteristiques', 'source', 'date_publication']


/tmp/ipykernel_461553/2197593358.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = train_raw.select_dtypes(include='object').columns.tolist()


## Étape 2 : Nettoyage

In [16]:
train = clean_dataframe(train_raw, is_train=True)
test  = clean_dataframe(test_raw,  is_train=False)

print(f"Train après nettoyage : {train.shape}")
print(f"Test  après nettoyage : {test.shape}")
print(f"\nPlage prix : {train['prix'].min():,.0f} → {train['prix'].max():,.0f} MRU")
print(f"Plage surface : {train['surface_m2'].min():.0f} → {train['surface_m2'].max():.0f} m²")
print(f"\nQuartiers uniques : {sorted(train['quartier'].unique())}")

Train après nettoyage : (1153, 12)
Test  après nettoyage : (289, 11)

Plage prix : 200,000 → 54,000,000 MRU
Plage surface : 20 → 2000 m²

Quartiers uniques : ['Arafat', 'Dar Naim', 'Ksar', 'Riyadh', 'Sebkha', 'Tevragh Zeina', 'Teyarett', 'Toujounine']


In [17]:
print("=== Valeurs manquantes TRAIN ===")
print(get_missing_summary(train))
print("\n=== Valeurs manquantes TEST ===")
print(get_missing_summary(test))

=== Valeurs manquantes TRAIN ===
                  nb_manquant  pct_manquant
nb_sdb                    833         72.25
caracteristiques          157         13.62
nb_chambres                14          1.21
id                          0          0.00
surface_m2                  0          0.00
prix                        0          0.00
titre                       0          0.00
nb_salons                   0          0.00
quartier                    0          0.00
description                 0          0.00
source                      0          0.00
date_publication            0          0.00

=== Valeurs manquantes TEST ===
                  nb_manquant  pct_manquant
nb_sdb                    216         74.74
caracteristiques           39         13.49
nb_chambres                 8          2.77
nb_salons                   2          0.69
surface_m2                  0          0.00
id                          0          0.00
titre                       0          0.00
quartier  

In [18]:
# Sauvegarder pour les notebooks suivants
train.to_csv('../data/processed/train_clean.csv', index=False)
test.to_csv('../data/processed/test_clean.csv',  index=False)
print("Données nettoyées sauvegardées.")

Données nettoyées sauvegardées.
